# Task 2 - Recommendation System Development

This notebook develops recommendation-system analysis for the Commercial Director scenario, focusing on customer value, product performance and purchase behaviour.

## 1. Import the required libraries

Pandas and NumPy support data preparation. Matplotlib and Seaborn are included for later charts, and Scikit-Learn's cosine similarity will support a collaborative recommendation approach.

In [1]:
# Import libraries for tables, calculations and charts.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import cosine similarity for a later collaborative-filtering recommender.
from sklearn.metrics.pairwise import cosine_similarity

## 2. Load the allocated datasets

Load the assigned customer, product and transaction CSV files from the `Datasets` folder. Keeping these as separate DataFrames makes each source easier to inspect before joining.

In [2]:
# Set paths to the three allocated CSV files.
customers_path = "../Datasets/Customers_members.csv"
products_path = "../Datasets/Products_overview.csv"
transactions_path = "../Datasets/Transactions_history.csv"

try:
    # Load each source dataset into its own DataFrame.
    customers_raw = pd.read_csv(customers_path)
    products_raw = pd.read_csv(products_path)
    transactions_raw = pd.read_csv(transactions_path)
    print("Customer, product and transaction datasets loaded successfully.")
except FileNotFoundError as error:
    print(f"A dataset file could not be found: {error.filename}")
    print("Check the CSV filenames and confirm that the notebook is in the Ca1-folder directory.")

Customer, product and transaction datasets loaded successfully.


## 3. Inspect the source datasets

Check each dataset's dimensions, sample rows, column names and data types before combining them. This helps confirm the join keys and the meaning of the transaction measures.

In [3]:
# Display dataset dimensions and the first five rows for each source.
for dataset_name, dataset in {
    "Customers": customers_raw,
    "Products": products_raw,
    "Transactions": transactions_raw
}.items():
    print(f"{dataset_name} dataset shape: {dataset.shape}")
    display(dataset.head())

# Inspect column names, data types and non-null counts.
print("Customers dataset information:")
customers_raw.info()

print("Products dataset information:")
products_raw.info()

print("Transactions dataset information:")
transactions_raw.info()

Customers dataset shape: (1000, 5)


,CustomerID,AgeGroup,Region,LoyaltyLevel,MembershipYears
0,1,36-50,East,Bronze,9
1,2,26-35,East,Bronze,4
2,3,18-25,East,Silver,10
3,4,36-50,East,Bronze,14
4,5,26-35,North,Bronze,7


Products dataset shape: (200, 6)


,ProductID,Category,Brand,Price,Rating,PopularityScore
0,1,Electronics,TechMax,474.45,4.54,3
1,2,Gaming,SmartBuy,285.11,3.16,2
2,3,books,HomePro,370.79,3.01,1
3,4,Home,TechMax,416.38,3.38,1
4,5,Books,TechMax,62.37,3.42,1


Transactions dataset shape: (19166, 5)


,CustomerID,ProductID,PurchaseCount,LastPurchaseDays,Spend
0,1,82,3,162,104.01
1,2,7,4,276,387.48
2,3,170,4,215,1507.40
3,3,9,1,251,341.40
4,4,64,3,354,1250.40


Customers dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   CustomerID       1000 non-null   int64
 1   AgeGroup         1000 non-null   str  
 2   Region           1000 non-null   str  
 3   LoyaltyLevel     1000 non-null   str  
 4   MembershipYears  1000 non-null   int64
dtypes: int64(2), str(3)
memory usage: 39.2 KB
Products dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ProductID        200 non-null    int64  
 1   Category         200 non-null    str    
 2   Brand            200 non-null    str    
 3   Price            200 non-null    float64
 4   Rating           185 non-null    float64
 5   PopularityScore  200 non-null    int64  
dtypes: float64(2), int64(2),

## 4. Assess data quality and key integrity

Summarise missing values, exact duplicate rows and duplicate entity keys. Transaction duplicates are reported for review rather than removed automatically because the file has no transaction ID to distinguish accidental duplicates from repeated purchases.

In [4]:
# Summarise missing cells and fully duplicated rows in each source dataset.
data_quality_summary = pd.DataFrame({
    "Missing Cells": [
        customers_raw.isnull().sum().sum(),
        products_raw.isnull().sum().sum(),
        transactions_raw.isnull().sum().sum()
    ],
    "Exact Duplicate Rows": [
        customers_raw.duplicated().sum(),
        products_raw.duplicated().sum(),
        transactions_raw.duplicated().sum()
    ]
}, index=["Customers", "Products", "Transactions"])

data_quality_summary

,Missing Cells,Exact Duplicate Rows
Customers,0,0
Products,15,0
Transactions,0,384


In [ ]:
# Check whether the entity keys are unique in their respective datasets.
key_duplicate_summary = pd.Series({
    "Duplicate CustomerID values": customers_raw["CustomerID"].duplicated().sum(),
    "Duplicate ProductID values": products_raw["ProductID"].duplicated().sum()
}, name="Duplicate key count")

print("Duplicate entity keys:")
display(key_duplicate_summary)

# Check that each transaction references an existing customer and product.
unmatched_customer_count = (~transactions_raw["CustomerID"].isin(customers_raw["CustomerID"])).sum()
unmatched_product_count = (~transactions_raw["ProductID"].isin(products_raw["ProductID"])).sum()

print(f"Transactions with an unknown CustomerID: {unmatched_customer_count}")
print(f"Transactions with an unknown ProductID: {unmatched_product_count}")

## 5. Integrate the datasets

Create clean working copies, standardize product category capitalization, then enrich transaction records with customer and product details. The join validation checks that each transaction links to one customer and one product.

In [ ]:
# Create working copies and standardize category labels for consistent grouping.
customers = customers_raw.copy()
products = products_raw.copy()
transactions = transactions_raw.copy()

products["Category"] = products["Category"].str.strip().str.title()

# Add customer and product details to each transaction.
transactions_enriched = transactions.merge(
    customers,
    on="CustomerID",
    how="left",
    validate="many_to_one"
).merge(
    products,
    on="ProductID",
    how="left",
    validate="many_to_one"
)

print(f"Transactions before integration: {len(transactions)}")
print(f"Transactions after integration: {len(transactions_enriched)}")
print(f"Integrated dataset shape: {transactions_enriched.shape}")
display(transactions_enriched.head())

## 6. Create the customer-product interaction matrix

The matrix aggregates `PurchaseCount` for each customer-product pair. A zero means no recorded purchase interaction in the transaction data. A second matrix sums `Spend` for later customer-value analysis.

In [ ]:
# Aggregate repeated customer-product records into total purchase counts.
interaction_matrix = transactions.pivot_table(
    index="CustomerID",
    columns="ProductID",
    values="PurchaseCount",
    aggfunc="sum",
    fill_value=0
)

# Retain a spend matrix for customer and product value analysis.
spend_matrix = transactions.pivot_table(
    index="CustomerID",
    columns="ProductID",
    values="Spend",
    aggfunc="sum",
    fill_value=0
)

print(f"Interaction matrix shape (customers, products): {interaction_matrix.shape}")
print(f"Number of recorded customer-product interactions: {(interaction_matrix > 0).sum().sum()}")
display(interaction_matrix.head())